# Options Earnings Analysis

Finds US stock options expiring the **same week as an earnings announcement**, where:
- Earnings fall on a **Friday**
- Earnings are announced **before market open (BMO) or after market close (AMC)** — not during trading hours
- Short-leg delta is in the **0.20–0.30** range
- Greeks are calculated via Black-Scholes (delta, gamma, vega, theta)
- Each option row is paired with the **underlying spot price** and its **timestamp**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import norm
from datetime import datetime, timedelta, date
import time

print('Libraries loaded OK')

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
RISK_FREE_RATE = 0.045          # ~4.5% annualised T-bill yield
DELTA_MIN      = 0.20           # short-leg delta lower bound
DELTA_MAX      = 0.30           # short-leg delta upper bound
DAYS_LOOKAHEAD = 21             # how many calendar days forward to look for earnings

# Universe: add or remove tickers here
TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META',
    'NVDA', 'TSLA', 'AMD',   'NFLX', 'CRM',
    'ADBE', 'INTC', 'QCOM',  'MU',   'AVGO',
]

TODAY = date.today()
print(f'Today: {TODAY}  |  Looking {DAYS_LOOKAHEAD} days forward  |  Delta window: {DELTA_MIN}–{DELTA_MAX}')

In [ ]:
# ── Black-Scholes Greeks ──────────────────────────────────────────────────────
def bs_greeks(S: float, K: float, T: float, r: float, sigma: float, opt: str = 'call') -> dict:
    """
    Return Black-Scholes delta, gamma, vega, theta for a European option.
    T  : time to expiry in years
    opt: 'call' or 'put'
    """
    if T <= 0 or sigma <= 0 or S <= 0:
        return dict(delta=np.nan, gamma=np.nan, vega=np.nan, theta=np.nan)

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if opt == 'call':
        delta = norm.cdf(d1)
        theta = (
            -S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
            - r * K * np.exp(-r * T) * norm.cdf(d2)
        ) / 365
    else:  # put
        delta = norm.cdf(d1) - 1
        theta = (
            -S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
            + r * K * np.exp(-r * T) * norm.cdf(-d2)
        ) / 365

    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega  = S * norm.pdf(d1) * np.sqrt(T) / 100   # per 1% move in vol

    return dict(delta=round(delta, 4), gamma=round(gamma, 6),
                vega=round(vega, 4),   theta=round(theta, 4))


# quick sanity check
test = bs_greeks(S=200, K=220, T=30/365, r=0.045, sigma=0.30, opt='call')
print('Sanity check (OTM call, delta should be ~0.20–0.25):', test)

In [ ]:
# ── Earnings helpers ─────────────────────────────────────────────────────────

def get_next_earnings(ticker_obj) -> tuple[date | None, str]:
    """
    Returns (earnings_date, timing) where timing is 'BMO', 'AMC', or 'Unknown'.
    Uses yfinance .calendar which returns the next scheduled earnings date.
    BMO/AMC detection: yfinance surfaces a datetime with time component when
    earnings time is known; otherwise we get a bare date.
    """
    try:
        cal = ticker_obj.calendar
        if not cal or 'Earnings Date' not in cal:
            return None, 'Unknown'

        raw = cal['Earnings Date']
        # yfinance returns a list with 1-2 dates (range)
        if isinstance(raw, (list, tuple)):
            raw = raw[0]

        if isinstance(raw, datetime):
            earnings_date = raw.date()
            hour = raw.hour
        elif isinstance(raw, date):
            earnings_date = raw
            hour = None
        else:
            earnings_date = pd.Timestamp(raw).date()
            hour = None

        # Determine BMO / AMC from hour if available
        if hour is None:
            timing = 'Unknown'
        elif hour < 9 or (hour == 9 and raw.minute < 30):
            timing = 'BMO'
        elif hour >= 16:
            timing = 'AMC'
        else:
            timing = 'During Market'

        return earnings_date, timing

    except Exception as e:
        return None, 'Unknown'


def is_friday(d: date) -> bool:
    return d.weekday() == 4   # Monday=0 … Friday=4


def week_start(d: date) -> date:
    """Monday of the ISO week containing d."""
    return d - timedelta(days=d.weekday())


print('Earnings helpers defined.')

In [ ]:
# ── Per-ticker options fetch ──────────────────────────────────────────────────

def fetch_options_for_ticker(ticker: str) -> pd.DataFrame | None:
    """
    For a given ticker:
    1. Check earnings date — must be a Friday within DAYS_LOOKAHEAD
    2. Timing must be BMO or AMC (not during market)
    3. Fetch all option chains expiring the same calendar week as earnings
    4. Compute Black-Scholes greeks using yfinance impliedVolatility
    5. Filter to short-leg delta window 0.20–0.30 (OTM puts & calls)
    Returns a DataFrame or None if criteria not met.
    """
    try:
        t = yf.Ticker(ticker)

        # ── 1. Earnings check ────────────────────────────────────────────────
        earnings_date, timing = get_next_earnings(t)

        if earnings_date is None:
            return None

        days_to_earnings = (earnings_date - TODAY).days
        if not (0 <= days_to_earnings <= DAYS_LOOKAHEAD):
            return None

        if not is_friday(earnings_date):
            return None

        if timing not in ('BMO', 'AMC', 'Unknown'):
            # 'During Market' is excluded
            return None

        # ── 2. Underlying spot price ─────────────────────────────────────────
        info = t.fast_info
        spot = info.last_price
        spot_ts = datetime.now()   # best available timestamp when we fetch

        if not spot or np.isnan(spot):
            return None

        # ── 3. Find expirations in the same week as earnings ─────────────────
        earnings_week = week_start(earnings_date)
        all_exps = t.options   # tuple of 'YYYY-MM-DD' strings

        target_exps = [
            exp for exp in all_exps
            if week_start(date.fromisoformat(exp)) == earnings_week
        ]

        if not target_exps:
            return None

        # ── 4. Fetch chains, compute greeks, filter by delta ─────────────────
        rows = []
        for exp_str in target_exps:
            exp_date = date.fromisoformat(exp_str)
            T = max((exp_date - TODAY).days, 0) / 365.0

            chain = t.option_chain(exp_str)

            for opt_type, df in [('call', chain.calls), ('put', chain.puts)]:
                for _, row in df.iterrows():
                    iv = row.get('impliedVolatility', np.nan)
                    if np.isnan(iv) or iv <= 0:
                        continue

                    greeks = bs_greeks(
                        S=spot, K=row['strike'], T=T,
                        r=RISK_FREE_RATE, sigma=iv, opt=opt_type
                    )

                    delta_abs = abs(greeks['delta'])
                    if not (DELTA_MIN <= delta_abs <= DELTA_MAX):
                        continue

                    rows.append({
                        'ticker':           ticker,
                        'earnings_date':    earnings_date,
                        'earnings_timing':  timing,
                        'expiration':       exp_date,
                        'option_type':      opt_type,
                        'strike':           row['strike'],
                        'bid':              row.get('bid', np.nan),
                        'ask':              row.get('ask', np.nan),
                        'mid':              round((row.get('bid', 0) + row.get('ask', 0)) / 2, 2),
                        'impliedVol':       round(iv, 4),
                        'openInterest':     row.get('openInterest', np.nan),
                        'volume':           row.get('volume', np.nan),
                        'delta':            greeks['delta'],
                        'gamma':            greeks['gamma'],
                        'vega':             greeks['vega'],
                        'theta':            greeks['theta'],
                        'spot_price':       round(spot, 4),
                        'spot_timestamp':   spot_ts,
                        'days_to_expiry':   (exp_date - TODAY).days,
                        'days_to_earnings': days_to_earnings,
                    })

        if not rows:
            return None

        return pd.DataFrame(rows)

    except Exception as e:
        print(f'  [{ticker}] ERROR: {e}')
        return None


print('fetch_options_for_ticker defined.')

In [ ]:
# ── Scan the full universe ────────────────────────────────────────────────────

results = []
skipped  = []

for ticker in TICKERS:
    print(f'Scanning {ticker}...', end=' ')
    df = fetch_options_for_ticker(ticker)
    if df is not None and len(df) > 0:
        results.append(df)
        print(f'{len(df)} contracts matched.')
    else:
        skipped.append(ticker)
        print('skipped (no earnings Friday match or no delta match).')
    time.sleep(0.4)   # be polite to yfinance

if results:
    all_options = pd.concat(results, ignore_index=True)
    print(f'\nTotal matched contracts: {len(all_options)}')
else:
    all_options = pd.DataFrame()
    print('\nNo matches found for any ticker in the universe.')

print(f'Tickers skipped: {skipped}')

In [ ]:
# ── Inspect results ───────────────────────────────────────────────────────────

if all_options.empty:
    print('No data to display.')
else:
    display_cols = [
        'ticker', 'earnings_date', 'earnings_timing',
        'expiration', 'option_type', 'strike',
        'bid', 'ask', 'mid', 'impliedVol',
        'delta', 'gamma', 'vega', 'theta',
        'spot_price', 'spot_timestamp',
        'days_to_expiry', 'days_to_earnings',
        'openInterest', 'volume',
    ]
    display(all_options[display_cols].sort_values(['ticker', 'expiration', 'option_type', 'strike']))

In [ ]:
# ── Analysis: summary per ticker ─────────────────────────────────────────────

if not all_options.empty:
    summary = (
        all_options
        .groupby(['ticker', 'earnings_date', 'earnings_timing', 'option_type'])
        .agg(
            contracts       = ('strike', 'count'),
            avg_delta       = ('delta', 'mean'),
            avg_iv          = ('impliedVol', 'mean'),
            avg_mid         = ('mid', 'mean'),
            avg_theta       = ('theta', 'mean'),
            total_oi        = ('openInterest', 'sum'),
        )
        .reset_index()
        .round(4)
    )
    print('Summary by ticker / earnings date / option type:')
    display(summary)

In [ ]:
# ── Export to CSV ─────────────────────────────────────────────────────────────

if not all_options.empty:
    out_path = f'options_earnings_{TODAY.isoformat()}.csv'
    all_options.to_csv(out_path, index=False)
    print(f'Saved {len(all_options)} rows → {out_path}')
else:
    print('Nothing to export.')